In [ ]:
import sys
print(sys.executable)

import torch, nltk, pickle
from torch import nn
from collections import Counter
from transformers import BatchEncoding, PretrainedConfig, PreTrainedModel
from transformers.modeling_outputs import CausalLMOutput

import numpy as np
import sys, time, os

/Users/filipn/Documents/dev/WASP-DL-NLP/env/bin/python


/Users/filipn/Documents/dev/WASP-DL-NLP/env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
###
### Part 1. Tokenization.
###

class A1Tokenizer:
    """A minimal implementation of a tokenizer similar to tokenizers in the HuggingFace library."""

    def __init__(self, str_to_int, int_to_str, tokenize_fun, model_max_length, pad_token,
                 unk_token, bos_token, eos_token):
        # store all values you need in order to implement __call__ below.
        self.str_to_int = str_to_int
        self.int_to_str = int_to_str
        self.tokenize_fun = tokenize_fun
        self.model_max_length = model_max_length # Needed for truncation.

        self.pad_token_id = str_to_int[pad_token] # Compulsory attribute.
        self.unk_token_id = str_to_int[unk_token]
        self.bos_token_id = str_to_int[bos_token]
        self.eos_token_id = str_to_int[eos_token]


    def __call__(self, texts, truncation=False, padding=False, return_tensors=None):
        """Tokenize the given texts and return a BatchEncoding containing the integer-encoded tokens.
           
           Args:
             texts:           The texts to tokenize.
             truncation:      Whether the texts should be truncated to model_max_length.
             padding:         Whether the tokenized texts should be padded on the right side.
             return_tensors:  If None, then return lists; if 'pt', then return PyTorch tensors.

           Returns:
             A BatchEncoding where the field `input_ids` stores the integer-encoded texts.
        """
        if return_tensors and return_tensors != 'pt':
            raise ValueError('Should be pt')
        
        # TODO: Your work here is to split the texts into words and map them to integer values.
        # 
        # - If `truncation` is set to True, the length of the encoded sequences should be 
        #   at most self.model_max_length.
        # - If `padding` is set to True, then all the integer-encoded sequences should be of the
        #   same length. That is: the shorter sequences should be "padded" by adding dummy padding
        #   tokens on the right side.
        # - If `return_tensors` is undefined, then the returned `input_ids` should be a list of lists.
        #   Otherwise, if `return_tensors` is 'pt', then `input_ids` should be a PyTorch 2D tensor.

        # Return a BatchEncoding where input_ids stores the result of the integer encoding.
        # Optionally, if you want to be 100% HuggingFace-compatible, you should also include an 
        # attention mask of the same shape as input_ids. In this mask, padding tokens correspond
        # to the the value 0 and real tokens to the value 1.
        encoded_texts = []
        for text in texts:
            word_tokens = self.tokenize_fun(text)
            token_ids = [self.bos_token_id]
            for word_token in word_tokens:
                token_id = self.str_to_int.get(word_token, self.unk_token_id)
                token_ids.append(token_id)
            token_ids.append(self.eos_token_id)
            if truncation and self.model_max_length is not None:
                token_ids = token_ids[:self.model_max_length]
                token_ids[-1] = self.eos_token_id
            encoded_texts.append(token_ids)

        if padding:
            longest_length = max(len(token_ids) for token_ids in encoded_texts)
            for token_ids in encoded_texts:
                number_of_padding_tokens = longest_length - len(token_ids)
                token_ids.extend([self.pad_token_id] * number_of_padding_tokens)

        attention_mask = [[0 if token_id == self.pad_token_id else 1 for token_id in token_ids] for token_ids in encoded_texts]

        if return_tensors == 'pt':
            encoded_texts = torch.tensor(encoded_texts)
            attention_mask = torch.tensor(attention_mask)

        return BatchEncoding({
            'input_ids': encoded_texts,
            'attention_mask': attention_mask
        })
        # return BatchEncoding({'input_ids': ...})

    def __len__(self):
        """Return the size of the vocabulary."""
        return len(self.str_to_int)
    
    def save(self, filename):
        """Save the tokenizer to the given file."""
        with open(filename, 'wb') as f:
            pickle.dump(self, f)

    @staticmethod
    def from_file(filename):
        """Load a tokenizer from the given file."""
        with open(filename, 'rb') as f:
            return pickle.load(f)


In [3]:
def lowercase_tokenizer(text):
    return [t.lower() for t in nltk.word_tokenize(text)]


def build_tokenizer(train_file, tokenize_fun=lowercase_tokenizer, max_voc_size=None, model_max_length=None,
                    pad_token='<PAD>', unk_token='<UNK>', bos_token='<BOS>', eos_token='<EOS>'):
    """ Build a tokenizer from the given file.

        Args:
             train_file:        The name of the file containing the training texts.
             tokenize_fun:      The function that maps a text to a list of string tokens.
             max_voc_size:      The maximally allowed size of the vocabulary.
             model_max_length:  Truncate texts longer than this length.
             pad_token:         The dummy string corresponding to padding.
             unk_token:         The dummy string corresponding to out-of-vocabulary tokens.
             bos_token:         The dummy string corresponding to the beginning of the text.
             eos_token:         The dummy string corresponding to the end the text.
    """
    # build the vocabulary, possibly truncating it to max_voc_size if that is specified.
    # Then return a tokenizer object (implemented below).
    special_tokens = [pad_token, unk_token, bos_token, eos_token]
    counter = Counter()
    with open(train_file, encoding='utf-8') as file:
        for line in file:
            text = line.strip()
            if text: counter.update(tokenize_fun(text))
    # print(counter)

    str_to_int = {}
    for token in special_tokens:
        str_to_int[token] = len(str_to_int)

    if max_voc_size is None:
        num_regular_tokens = None
    else:
        num_regular_tokens = max_voc_size - len(special_tokens)
        if num_regular_tokens < 0:
            raise ValueError("max_voc_size must be at least the number of special tokens")
    
    for token, _ in counter.most_common(num_regular_tokens):
          if token not in str_to_int:
              str_to_int[token] = len(str_to_int)

    # print(str_to_int)
    if max_voc_size: assert len(str_to_int) <= max_voc_size, "max_voc_size exeeded"

    int_to_str = {i: token for token, i in str_to_int.items()}

    return A1Tokenizer(
        str_to_int=str_to_int,
        int_to_str=int_to_str,
        tokenize_fun=tokenize_fun,
        model_max_length=model_max_length,
        pad_token=pad_token,
        unk_token=unk_token,
        bos_token=bos_token,
        eos_token=eos_token,
    )


print(lowercase_tokenizer("Let's test!!"))
tokenizer = build_tokenizer(train_file="./train.txt", max_voc_size=1000, model_max_length=100)
print(len(tokenizer))


test_texts = ['This is a test.', 'Another test.']
result = tokenizer(test_texts, return_tensors='pt', padding=True, truncation=True)
print(result)
for encoded_text in result["input_ids"]: # type: ignore
    decoded_tokens = [tokenizer.int_to_str[token_id.item()] for token_id in encoded_text]
    print(decoded_tokens)

tokenizer.save('tokenizer_file.pkl')

['let', "'s", 'test', '!', '!']
1000
{'input_ids': tensor([[  2,  35,  14,  11, 975,   6,   3],
        [  2, 155, 975,   6,   3,   0,   0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 0, 0]])}
['<BOS>', 'this', 'is', 'a', 'test', '.', '<EOS>']
['<BOS>', 'another', 'test', '.', '<EOS>', '<PAD>', '<PAD>']


In [12]:
from datasets import load_dataset

dataset = load_dataset('text', data_files={'train': "./train.txt", 'val': "./val.txt"})
dataset = dataset.filter(lambda x: x['text'].strip() != '')

print(len(dataset["train"]))
print(len(dataset["val"]))

from torch.utils.data import Subset
for sec in ['train', 'val']:
    dataset[sec] = Subset(dataset[sec], range(1000)) # type: ignore

print(len(dataset["train"]))
print(len(dataset["val"]))

147059
17874
1000
1000


In [14]:
from torch.utils.data import DataLoader

dl = DataLoader(dataset['train'], batch_size=1, shuffle=True) # type: ignore

for batch in dl:
    print(batch)
    break

{'text': ['Angolan Armed Forces']}


In [ ]:
###
### Part 3. Defining the model.
###

class A1RNNModelConfig(PretrainedConfig):
    """Configuration object that stores hyperparameters that define the RNN-based language model."""
    def __init__(self, vocab_size, embedding_size, hidden_size, **kwargs):
        super().__init__(**kwargs)
        self.vocab_size = vocab_size
        self.hidden_size = hidden_size
        self.embedding_size = embedding_size

class A1RNNModel(PreTrainedModel):
    """The neural network model that implements a RNN-based language model."""
    config_class = A1RNNModelConfig
    
    def __init__(self, config):
        super().__init__(config)
        self.embedding = ...
        self.rnn = ...
        self.unembedding = ...

        # Note: -100 is the value HuggingFace conventionally uses to refer to tokens
        # where we do not want to compute the loss.
        self.loss_func = torch.nn.CrossEntropyLoss(ignore_index=-100)


    def forward(self, input_ids, labels=None):
        """The forward pass of the RNN-based language model.
        
           Args:
             - input_ids:  The input tensor (2D), consisting of a batch of integer-encoded texts.
             - labels:     The reference tensor (2D), consisting of a batch of integer-encoded texts.
           Returns:
             A CausalLMOutput containing
               - logits:   The output tensor (3D), consisting of logits for all token positions for all vocabulary items.
               - loss:     The loss computed on this batch.               
        """
        embedded = ...
        rnn_out, _ = ...
        logits = ...
        if labels is not None:
            loss = ...

        return CausalLMOutput(logits=logits, loss=loss)